In [1]:
import os
import json
import subprocess

def convert_ktx2_to_png(ktx2_path, output_dir):
    """
    Converts a .ktx2 file to .png using ktx2ktx2.
    Args:
        ktx2_path (str): Path to the .ktx2 file.
        output_dir (str): Directory to save the converted .png file.
    Returns:
        str: Path to the converted .png file.
    """
    png_path = os.path.join(output_dir, os.path.basename(ktx2_path).replace(".ktx2", ".png"))
    subprocess.run(["ktx2ktx2", "--decode", ktx2_path, "-o", png_path], check=True)
    return png_path

def update_gltf_references(gltf_path, output_dir):
    """
    Updates the .gltf file to replace .ktx2 references with .png.
    Args:
        gltf_path (str): Path to the .gltf file.
        output_dir (str): Directory containing the converted .png files.
    Returns:
        str: Path to the updated .gltf file.
    """
    # Load the GLTF file
    with open(gltf_path, "r", encoding="utf-8") as f:
        gltf_data = json.load(f)
    
    # Update the image references
    for image in gltf_data.get("images", []):
        if image.get("uri", "").endswith(".ktx2"):
            png_name = os.path.basename(image["uri"]).replace(".ktx2", ".png")
            image["uri"] = png_name  # Update to .png reference

    # Save the updated GLTF file
    updated_gltf_path = os.path.join(output_dir, os.path.basename(gltf_path))
    with open(updated_gltf_path, "w", encoding="utf-8") as f:
        json.dump(gltf_data, f, indent=2)
    
    return updated_gltf_path

def process_gltf(glb_path, output_dir):
    """
    Processes a .glb file by converting .ktx2 textures to .png and updating references in the .gltf file.
    Args:
        glb_path (str): Path to the .glb file.
        output_dir (str): Directory to save the processed files.
    """
    # Extract the .glb file
    gltf_pipeline_command = ["/homes/dtrofimov/.npm-global/bin/gltf-pipeline", "-i", glb_path, "-o", os.path.join(output_dir, "output.gltf"), "--separate"]
    subprocess.run(gltf_pipeline_command, check=True)

    # Identify extracted files
    extracted_files = os.listdir(output_dir)
    ktx2_files = [f for f in extracted_files if f.endswith(".ktx2")]
    gltf_file = [f for f in extracted_files if f.endswith(".gltf")][0]

    # Convert all .ktx2 files to .png
    for ktx2_file in ktx2_files:
        convert_ktx2_to_png(os.path.join(output_dir, ktx2_file), output_dir)
    
    # Update the .gltf file references
    updated_gltf_path = update_gltf_references(os.path.join(output_dir, gltf_file), output_dir)

    # Optionally repackage into a .glb file
    final_glb_path = os.path.join(output_dir, "final_output.glb")
    subprocess.run(["gltf-pipeline", "-i", updated_gltf_path, "-o", final_glb_path], check=True)

    print(f"Processed GLB file saved at: {final_glb_path}")

In [2]:
# Example Usage
input_glb = "/vol/isy-rl/dtrofimov/data/aihabitat-try/0001fb06b075a743e6289236cf049df3ad5dfa9c.glb"
output_dir = "/vol/isy-rl/dtrofimov/data/aihabitat-try-render/"
os.makedirs(output_dir, exist_ok=True)
process_gltf(input_glb, output_dir)

Total: 14.072ms


Usage: ktx2ktx2 [options] [<infile> ...]

  infile       The source ktx file. The output is written to a file of the
               same name with the extension changed to '.ktx2'. If it is not
               specified input will be read from stdin and the converted texture
               written to stdout.

  Options are:

  -b, --rewritebado
               Rewrite bad orientation metadata. Some in-the-wild KTX files
               have orientation metadata with the key "KTXOrientation"
               instead of "KTXorientaion". This option will rewrite such
               bad metadata instead of dropping it.
  -o outfile, --output=outfile
               Name the output file outfile. If @e outfile is 'stdout', output
               will be written to stdout. If there is more than 1 infile,
               the command prints its usage message and exits.
  -f, --force  If the output file already exists, remove it and create a
               new file, without prompting for confirmation re

CalledProcessError: Command '['ktx2ktx2', '--decode', '/vol/isy-rl/dtrofimov/data/aihabitat-try-render/output1.ktx2', '-o', '/vol/isy-rl/dtrofimov/data/aihabitat-try-render/output1.png']' returned non-zero exit status 1.

In [3]:
!ls /vol/isy-rl/dtrofimov/data/aihabitat-try-render/

output0.ktx2  output1.ktx2  output2.ktx2  output.bin  output.gltf


In [8]:
!ktx2ktx2 --decode --help

Usage: ktx2ktx2 [options] [<infile> ...]

  infile       The source ktx file. The output is written to a file of the
               same name with the extension changed to '.ktx2'. If it is not
               specified input will be read from stdin and the converted texture
               written to stdout.

  Options are:

  -b, --rewritebado
               Rewrite bad orientation metadata. Some in-the-wild KTX files
               have orientation metadata with the key "KTXOrientation"
               instead of "KTXorientaion". This option will rewrite such
               bad metadata instead of dropping it.
  -o outfile, --output=outfile
               Name the output file outfile. If @e outfile is 'stdout', output
               will be written to stdout. If there is more than 1 infile,
               the command prints its usage message and exits.
  -f, --force  If the output file already exists, remove it and create a
               new file, without prompting for confirmation re

In [9]:
!toktx --help

Usage: toktx [options] <outfile> [<infile>.{jpg,png,pam,pgm,ppm} ...]

  <outfile>    The destination ktx file. Parent directories will be created
               and ".ktx" will appended if necessary. If it is '-' the
               output will be written to stdout.
  <infile>     One or more image files in .jpg, .png, .pam, .ppm, or .pgm
               format. Other formats can be readily converted to these formats
               using tools such as ImageMagick and XnView. infiles prefixed
               with '@' are read as text files listing actual file names to
               process with one file path per line. Paths must be absolute or
               relative to the current directory when toktx is run. If '@@'
               is used instead, paths must be absolute or relative to the
               location of the list file. File paths must be encoded in UTF-8.

  The target texture type (number of components in the output texture) is chosen
  via --target_type. Swizzling of the c